# Enveda CASMI 2026 — Spec2Smiles Kaggle submission

**Competition:** [enveda-CASMI26-molecule-id-mass-spectra](https://www.kaggle.com/competitions/enveda-CASMI26-molecule-id-mass-spectra)  
**Type:** Featured **Code Competition** (inference must run in the notebook; internet may be off at submit time).

This notebook produces `/kaggle/working/submission.csv` with columns `molecule_id,smiles` where `smiles` is **exactly 25** semicolon-separated candidates, in the same `molecule_id` order as `sample_submission.csv`.

**Baseline:** Transformer spectrum encoder → SELFIES/SMILES decoder from this repo (`src/spec2smiles`, scripts `predict_kaggle.py` / `train.py`, config `configs/enveda_cpu_smoke.yaml`).

### Attach on Kaggle (Add Data)

1. **Competition dataset** — auto-attached when you create a notebook from the competition:  
   `/kaggle/input/enveda-CASMI26-molecule-id-mass-spectra/` (folder name can vary; cells below glob if needed).
2. **Code bundle** (pick one):
   - This GitHub repo as a Kaggle Dataset, **or**
   - Dataset built from the repo's `kaggle/` folder (`spec2smiles` + configs + scripts).
3. **Checkpoint Dataset** — upload `artifacts/checkpoints/enveda_cpu_smoke.pt` (gitignored `*.pt`) as e.g. `enveda-smoke-ckpt`.

### Honest v1 modes

| `RUN_MODE` | Behavior |
|------------|----------|
| `checkpoint` (default) | Load pre-uploaded smoke checkpoint → predict → submit |
| `train_subset` | Short fine-tune / train on a small train parquet subset (GPU if available), then predict |


## Setup checklist (once)

1. **Push repo to GitHub** (from your machine; do not commit tokens):
   ```bash
   cd MassSpecGym
   git remote add origin https://github.com/<you>/MassSpecGym.git   # if needed
   git push -u origin main
   ```
2. **Kaggle Dataset from code** — New Dataset → upload zip of repo **or** only `kaggle/` + `notebooks/`, **or** link the GitHub repo.
3. **Checkpoint Dataset** — upload `enveda_cpu_smoke.pt` (local path `artifacts/checkpoints/enveda_cpu_smoke.pt` after `python scripts/train.py --config configs/enveda_cpu_smoke.yaml`).
4. **Notebook** — Competition → Code → New Notebook → Add Data (competition + code + checkpoint) → paste/upload this notebook → **Save Version** → **Submit to Competition**.

See also `notebooks/README.md` in the repo.


## 1. Install light deps (rdkit, selfies; torch usually preinstalled on Kaggle)


In [ ]:
import importlib
import subprocess
import sys

def ensure(pkg, import_name=None):
    name = import_name or pkg
    try:
        importlib.import_module(name)
        print(f"OK {name}")
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        importlib.import_module(name)
        print(f"OK {name} (installed)")

# Torch / numpy / pandas / pyyaml / tqdm are typically present on Kaggle GPU/CPU images.
for pkg, mod in [
    ("pyyaml", "yaml"),
    ("tqdm", "tqdm"),
    ("selfies", "selfies"),
]:
    ensure(pkg, mod)

try:
    import rdkit  # noqa: F401
    print("OK rdkit")
except ImportError:
    for candidate in ("rdkit", "rdkit-pypi"):
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", candidate])
            import rdkit  # noqa: F401
            print(f"OK rdkit via {candidate}")
            break
        except Exception as e:
            print(f"skip {candidate}: {e}")

import torch
print("torch", torch.__version__, "cuda=", torch.cuda.is_available())


## 2. Resolve competition data, code bundle, and checkpoint paths


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

COMP_SLUG = "enveda-CASMI26-molecule-id-mass-spectra"
KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
IS_KAGGLE = KAGGLE_INPUT.exists()

LOCAL_ROOT = Path("/workspace/MassSpecGym")
if not LOCAL_ROOT.exists():
    LOCAL_ROOT = Path.cwd()
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "spec2smiles").is_dir() or (p / "kaggle" / "spec2smiles").is_dir():
            LOCAL_ROOT = p
            break

def first_existing(candidates):
    for c in candidates:
        if c is not None and Path(c).exists():
            return Path(c)
    return None

def find_comp_dir():
    # Competition files: sample_submission.csv + test.parquet (+ train.parquet)
    candidates = []
    if IS_KAGGLE:
        candidates.append(KAGGLE_INPUT / COMP_SLUG)
        for d in sorted(KAGGLE_INPUT.glob("*")):
            if not d.is_dir():
                continue
            if (d / "sample_submission.csv").exists() and (
                (d / "test.parquet").exists() or list(d.glob("test*.parquet"))
            ):
                candidates.append(d)
    candidates.append(LOCAL_ROOT / "data" / "kaggle" / "enveda-casmi26")
    found = first_existing(candidates)
    if found is None:
        raise FileNotFoundError(
            f"Could not find competition data under {KAGGLE_INPUT} or {LOCAL_ROOT}/data/kaggle/"
        )
    return found

def find_code_root():
    # Directory with src/spec2smiles, kaggle/spec2smiles, or flat spec2smiles/
    cands = []
    if IS_KAGGLE:
        for d in sorted(KAGGLE_INPUT.iterdir()):
            if not d.is_dir():
                continue
            if (d / "src" / "spec2smiles").is_dir():
                cands.append(d)
            if (d / "kaggle" / "spec2smiles").is_dir():
                cands.append(d / "kaggle")
            if (d / "spec2smiles").is_dir() and (d / "configs").is_dir():
                cands.append(d)
    cands.extend([LOCAL_ROOT, LOCAL_ROOT / "kaggle"])
    for c in cands:
        if (c / "src" / "spec2smiles").is_dir() or (c / "spec2smiles").is_dir():
            return c
    raise FileNotFoundError("Attach the MassSpecGym / kaggle code Dataset, or run from the repo root.")

def find_checkpoint():
    names = ["enveda_cpu_smoke.pt", "spec2smiles_cpu_subset.pt", "smoke_spec2smiles_best.pt"]
    cands = []
    if IS_KAGGLE:
        for d in KAGGLE_INPUT.rglob("*.pt"):
            cands.append(d)
    for name in names:
        cands.append(LOCAL_ROOT / "artifacts" / "checkpoints" / name)
        cands.append(LOCAL_ROOT / "kaggle" / "checkpoints" / name)
    cands.append(KAGGLE_WORKING / "checkpoints" / "enveda_cpu_smoke.pt")
    for name in names:
        hit = first_existing([c for c in cands if Path(c).name == name])
        if hit:
            return hit
    return first_existing(cands)

COMP_DIR = find_comp_dir()
CODE_ROOT = find_code_root()
CKPT_PATH = find_checkpoint()

if (CODE_ROOT / "src" / "spec2smiles").is_dir():
    sys.path.insert(0, str(CODE_ROOT / "src"))
    PKG_HINT = "src/spec2smiles"
elif (CODE_ROOT / "spec2smiles").is_dir():
    sys.path.insert(0, str(CODE_ROOT))
    PKG_HINT = "spec2smiles (kaggle bundle)"
else:
    raise RuntimeError(f"No spec2smiles under {CODE_ROOT}")

CONFIG_CANDIDATES = [
    CODE_ROOT / "configs" / "enveda_cpu_smoke.yaml",
    CODE_ROOT / "kaggle" / "configs" / "enveda_cpu_smoke.yaml",
    LOCAL_ROOT / "configs" / "enveda_cpu_smoke.yaml",
    LOCAL_ROOT / "kaggle" / "configs" / "enveda_cpu_smoke.yaml",
]
CONFIG_PATH = first_existing(CONFIG_CANDIDATES)

SAMPLE_CSV = COMP_DIR / "sample_submission.csv"
TEST_PARQUET = COMP_DIR / "test.parquet"
if not TEST_PARQUET.exists():
    alts = list(COMP_DIR.glob("test*.parquet"))
    TEST_PARQUET = alts[0] if alts else TEST_PARQUET
TRAIN_PARQUET = COMP_DIR / "train.parquet"

print("IS_KAGGLE     ", IS_KAGGLE)
print("COMP_DIR      ", COMP_DIR)
print("CODE_ROOT     ", CODE_ROOT, f"({PKG_HINT})")
print("CONFIG_PATH   ", CONFIG_PATH)
print("CKPT_PATH     ", CKPT_PATH)
print("SAMPLE_CSV    ", SAMPLE_CSV.exists(), SAMPLE_CSV)
print("TEST_PARQUET  ", TEST_PARQUET.exists(), TEST_PARQUET)
print("TRAIN_PARQUET ", TRAIN_PARQUET.exists())


## 3. Run mode

- **`checkpoint`**: load `enveda_cpu_smoke.pt` (or first found `.pt`) and predict — recommended for first submit.
- **`train_subset`**: short train / fine-tune on a capped train subset, write a working checkpoint, then predict.


In [ ]:
# --- user knobs ---
RUN_MODE = "checkpoint"  # "checkpoint" | "train_subset"
TOP_K = 25
FILL_SMILES = "CCO"
BEAM_SIZE = 5
BATCH_SIZE = 8
# train_subset caps (keep small for Code Comp time limits)
MAX_TRAIN_SAMPLES = 2000
MAX_VAL_SAMPLES = 200
MAX_STEPS = 150
EPOCHS = 1
# Optional debug: limit test spectra (None = all). Use a small int for a dry run.
MAX_SPECTRA = None

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("RUN_MODE", RUN_MODE, "DEVICE", DEVICE)


## 4. Optional short train / fine-tune


In [ ]:
from pathlib import Path

OUT_CKPT_DIR = Path("/kaggle/working/checkpoints") if IS_KAGGLE else (LOCAL_ROOT / "artifacts" / "checkpoints")
OUT_CKPT_DIR.mkdir(parents=True, exist_ok=True)
TRAINED_CKPT = OUT_CKPT_DIR / "enveda_casmi26_notebook.pt"

if RUN_MODE == "train_subset":
    if CONFIG_PATH is None:
        raise FileNotFoundError("Need configs/enveda_cpu_smoke.yaml in the code Dataset")
    if not TRAIN_PARQUET.exists():
        raise FileNotFoundError(f"train.parquet not found under {COMP_DIR}")

    from spec2smiles.train import load_config, train_loop

    cfg = load_config(str(CONFIG_PATH))
    cfg["device"] = DEVICE
    cfg.setdefault("data", {})
    cfg["data"]["source"] = "enveda"
    cfg["data"]["enveda_dir"] = str(COMP_DIR)
    cfg["data"]["max_train_samples"] = MAX_TRAIN_SAMPLES
    cfg["data"]["max_val_samples"] = MAX_VAL_SAMPLES
    cfg.setdefault("train", {})
    cfg["train"]["epochs"] = EPOCHS
    cfg["train"]["max_steps"] = MAX_STEPS
    cfg["train"]["batch_size"] = min(BATCH_SIZE, int(cfg["train"].get("batch_size", BATCH_SIZE)))
    cfg["train"]["num_workers"] = 0
    cfg["train"]["checkpoint_dir"] = str(OUT_CKPT_DIR)
    cfg["train"]["checkpoint_name"] = TRAINED_CKPT.name

    print("Training with", {k: cfg["data"].get(k) for k in ("enveda_dir", "max_train_samples", "decode_mode")})
    ckpt = train_loop(cfg, smoke=False)
    CKPT_PATH = Path(ckpt)
    print("Trained checkpoint:", CKPT_PATH)
elif RUN_MODE == "checkpoint":
    if CKPT_PATH is None or not Path(CKPT_PATH).exists():
        raise FileNotFoundError(
            "No checkpoint found. Upload enveda_cpu_smoke.pt as a Kaggle Dataset, "
            "or set RUN_MODE='train_subset'."
        )
    print("Using checkpoint:", CKPT_PATH)
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE!r}")


## 5. Predict → `/kaggle/working/submission.csv`


In [ ]:
from collections import defaultdict
from pathlib import Path

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from spec2smiles.chem_utils import is_valid_smiles
from spec2smiles.data import SpectrumDataset, collate_batch, load_enveda_parquet_rows
from spec2smiles.decode import beam_candidates_one
from spec2smiles.train import get_decode_mode, load_checkpoint, load_config, resolve_device

assert CONFIG_PATH is not None, "config yaml required"
assert CKPT_PATH is not None and Path(CKPT_PATH).exists(), "checkpoint required"

cfg = load_config(str(CONFIG_PATH))
device = resolve_device(DEVICE)
model, vocab, ckpt_cfg = load_checkpoint(Path(CKPT_PATH), device=device)

data_cfg = dict(cfg.get("data", {}))
data_cfg.update({
    k: v for k, v in ckpt_cfg.get("data", {}).items()
    if k in ("max_peaks", "max_smiles_len", "mz_bins", "mz_max", "decode_mode")
})
decode_mode = get_decode_mode(ckpt_cfg) if ckpt_cfg.get("data") else get_decode_mode(cfg)
infer_cfg = {**cfg.get("infer", {}), **ckpt_cfg.get("infer", {})}
beam_size = BEAM_SIZE if BEAM_SIZE is not None else int(infer_cfg.get("beam_size", 5))
max_len = int(infer_cfg.get("max_len", data_cfg.get("max_smiles_len", 128)))

sample = pd.read_csv(SAMPLE_CSV)
molecule_order = sample["molecule_id"].astype(str).tolist()

test_rows = load_enveda_parquet_rows(
    TEST_PARQUET,
    split="test",
    max_samples=MAX_SPECTRA,
)
if not test_rows:
    raise RuntimeError("No test spectra loaded")

for r in test_rows:
    if not r.get("smiles"):
        r["smiles"] = FILL_SMILES
        r["target"] = FILL_SMILES

ds = SpectrumDataset(
    test_rows,
    vocab=vocab,
    max_peaks=int(data_cfg.get("max_peaks", 100)),
    max_smiles_len=int(data_cfg.get("max_smiles_len", 128)),
    mz_bins=int(data_cfg.get("mz_bins", 5000)),
    mz_max=float(data_cfg.get("mz_max", 2000.0)),
)
loader = DataLoader(
    ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_batch,
)

agg = defaultdict(list)
mol_ids = [str(r["molecule_id"]) for r in test_rows]

def rank_candidates(scored, top_k, fill):
    best = {}
    for sc, smi in scored:
        smi = (smi or "").strip()
        if not smi:
            continue
        if smi not in best or sc > best[smi]:
            best[smi] = sc
    ranked = sorted(best.items(), key=lambda kv: kv[1], reverse=True)
    valid = [s for s, _ in ranked if is_valid_smiles(s)]
    invalid = [s for s, _ in ranked if not is_valid_smiles(s)]
    ordered = valid + invalid
    seen = set()
    uniq = []
    for s in ordered:
        if s not in seen:
            seen.add(s)
            uniq.append(s)
    while len(uniq) < top_k:
        uniq.append(fill)
    return uniq[:top_k]

model.eval()
idx0 = 0
with torch.no_grad():
    for batch in tqdm(loader, desc="predict"):
        B = batch["mz_bin_ids"].size(0)
        mz = batch["mz_bin_ids"].to(device)
        inten = batch["intensity"].to(device)
        mzn = batch["mz_norm"].to(device)
        pmask = batch["peak_mask"].to(device)
        pmz = batch.get("precursor_mz")
        if pmz is not None:
            pmz = pmz.to(device)
        for b in range(B):
            mid = mol_ids[idx0 + b]
            pmz_b = pmz[b : b + 1] if pmz is not None else None
            cands = beam_candidates_one(
                model,
                vocab,
                mz[b : b + 1],
                inten[b : b + 1],
                mzn[b : b + 1],
                pmask[b : b + 1],
                beam_size=beam_size,
                max_len=max_len,
                precursor_mz=pmz_b,
                decode_mode=decode_mode,
            )
            agg[mid].extend(cands)
        idx0 += B

rows_out = []
for mid in molecule_order:
    cands = rank_candidates(agg.get(mid, []), top_k=TOP_K, fill=FILL_SMILES)
    rows_out.append({"molecule_id": mid, "smiles": ";".join(cands)})

OUT_PATH = (KAGGLE_WORKING / "submission.csv") if IS_KAGGLE else (LOCAL_ROOT / "artifacts" / "submissions" / "enveda_notebook.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sub = pd.DataFrame(rows_out, columns=["molecule_id", "smiles"])
sub.to_csv(OUT_PATH, index=False)

n_ok = sum(1 for s in sub["smiles"] if len(str(s).split(";")) == TOP_K)
order_ok = sub["molecule_id"].astype(str).tolist() == molecule_order
covered = sum(1 for mid in molecule_order if mid in agg and agg[mid])
print(f"Wrote {OUT_PATH}")
print(f"rows={len(sub)} sample={len(molecule_order)} exact_{TOP_K}_slots={n_ok} order_ok={order_ok}")
print(f"molecules_with_preds={covered} spectra={len(test_rows)} beam={beam_size} decode_mode={decode_mode}")
sub.head(3)


## 6. Submit

1. Confirm `/kaggle/working/submission.csv` exists (Output tab after Save Version).
2. **Save Version** → version type **Save & Run All** (required for Code Competitions).
3. When the run succeeds, **Submit** that version's `submission.csv` to the competition.

Local equivalent (repo root, after smoke train):

```bash
python scripts/predict_kaggle.py \
  --checkpoint artifacts/checkpoints/enveda_cpu_smoke.pt \
  --config configs/enveda_cpu_smoke.yaml \
  --data-dir data/kaggle/enveda-casmi26 \
  --output artifacts/submissions/enveda_smoke.csv
```

**Note:** This smoke model is a starting baseline, not a SOTA entry. Increase `MAX_TRAIN_SAMPLES` / epochs on GPU, or swap in a stronger checkpoint Dataset, for real scores.
